# Kamień–papier–nożyce na UNIHIKER K10: trening BEZ własnych zdjęć

Ta wersja nie potrzebuje zdjęć z kamery K10. Model uczy się z dwóch rodzajów danych, które notebook
sam pobiera:

**A. Sztuczne sceny** (jak dotąd, ale bogatsze):

1. **Dłonie wycięte z gotowych zbiorów:** zdjęcia na zielonym tle z Kaggle oraz dłonie renderowane
   komputerowo na białym tle (różne kolory skóry, dłoń widziana z różnych stron).
2. **Wklejone na losowe tła:** zdjęcia wnętrz, fragmenty zdjęć roślin i tła generowane.
3. **Ustawione jak przed kamerą K10:** ręka wchodzi do kadru od dołu (czasem z boku), ma różny rozmiar,
   położenie, obrót i pochylenie. Dłoń dostaje światło pasujące do tła, miękką krawędź i czasem cień.
4. **Kamera K10 jest udawana:** balans bieli, gamma, ekspozycja, rozmycie (także poruszenie), szum,
   winietowanie, zmniejszenie 240→96 przez uśrednianie bloków (jak na płytce) i redukcja kolorów do RGB565.

**B. Prawdziwe zdjęcia ludzi pokazujących gesty** (zbiór HaGRID): tysiące różnych osób, pokoi i oświetleń.
Z każdego zdjęcia brany jest kwadrat wokół dłoni, który przechodzi przez tę samą „kamerę K10”.

**C. Kadry bez dłoni** (klasa `none`): wycinki tych samych zdjęć HaGRID z miejsc, w których dłoni nie ma
(twarz, ubranie, pokój), oraz sztuczne sceny bez wklejonej dłoni. Dzięki temu K10 pokazuje „BRAK REKI”,
kiedy nikt nic nie pokazuje, zamiast zgadywać gest. Wyłącza to ustawienie `KLASA_BRAK = False`.

Na koniec model jest sprawdzany na **prawdziwych zdjęciach, których nie widział w treningu**
(inne osoby z HaGRID i osobny zbiór zdjęć z kamerki internetowej). To ten wynik mówi, czego się spodziewać
na płytce; wynik na sztucznych scenach jest zawsze zbyt optymistyczny.

Dalej jest jak w zwykłym notebooku: MobileNetV2 0.35, 96×96, kwantyzacja INT8 (z dopasowaniem ostatniej warstwy
do modelu INT8, sekcja 9b) i plik `model_data.h`.

**Kaggle:**
- Settings → *Internet: On* (wagi ImageNet i pobieranie zbiorów),
- dodaj dataset `drgfreeman/rockpaperscissors` (Add Input),
- GPU T4 przyspiesza trening, ale nie jest wymagane.

In [ ]:
import os, sys, glob, math, random, subprocess, importlib, json, time, shutil, hashlib, urllib.request
import multiprocessing as mp
import numpy as np
print("NumPy", np.__version__)

# Na Kaggle zdarza się, że preinstalowany OpenCV jest skompilowany pod NumPy 2.x, a środowisko ma
# NumPy 1.x (błąd "numpy._core.multiarray failed to import"). Wtedy doinstalowujemy obok wersję
# OpenCV zgodną z obiema wersjami NumPy, bez ruszania NumPy ani TensorFlow.
def importuj_cv2():
    try:
        import cv2
        return cv2
    except ImportError as e:
        print("OpenCV nie pasuje do tej wersji NumPy:", e)
        print("Instaluję zgodny OpenCV (opencv-python-headless 4.10.0.84) do /tmp/cv2fix ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                        "--target", "/tmp/cv2fix", "opencv-python-headless==4.10.0.84"], check=True)
        for m in [m for m in list(sys.modules) if m == "cv2" or m.startswith("cv2.")]:
            del sys.modules[m]
        sys.path.insert(0, "/tmp/cv2fix")
        importlib.invalidate_caches()
        import cv2
        return cv2

cv2 = importuj_cv2()
import tensorflow as tf
import keras
from keras import layers
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "| OpenCV", cv2.__version__)

## 0. Google Colab: zapis na Dysku Google

Ta komórka działa tylko w Colabie (na Kaggle i w lokalnym Jupyterze nic nie robi). Podłącza Twój
Dysk Google i używa folderu **`Mój dysk/K10`**, który przetrwa zamknięcie środowiska:

- **Zbiór danych:**
  - przy pierwszym uruchomieniu notebook pobiera go z Kaggle i zapisuje na Dysku jako `K10/rockpaperscissors.zip`,
  - przy kolejnych rozpakowuje go z Dysku (zamiast pobierać), co trwa kilkanaście sekund,
  - jeśli wolisz, wgraj na Dysk do `K10/` własny ZIP albo folder `rps-cv-images` (z `paper/`, `rock/`, `scissors/`), a notebook użyje go zamiast pobierania.
- **Wyniki:** `model_data.h`, `rps_int8.tflite` i wytrenowany model `model.keras` lądują w `K10/wyniki/`.

**GPU:** *Środowisko wykonawcze → Zmień typ środowiska wykonawczego → T4 GPU*.

In [ ]:
import shutil
W_COLAB = "google.colab" in sys.modules
DANE_COLAB = OUT_DIR_COLAB = None
KLASY_ZBIORU = ("paper", "rock", "scissors")

def folder_z_klasami(baza):
    for root, dirs, _ in os.walk(baza):
        dirs[:] = [d for d in dirs if not d.startswith(".")]
        if all(os.path.isdir(os.path.join(root, c)) for c in KLASY_ZBIORU):
            return root
    return None

if W_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJEKT = "/content/drive/MyDrive/K10"
    os.makedirs(PROJEKT, exist_ok=True)
    OUT_DIR_COLAB = os.path.join(PROJEKT, "wyniki")
    os.makedirs(OUT_DIR_COLAB, exist_ok=True)
    DANE = "/content/dane"                         # szybki dysk lokalny Colaba

    juz_rozpakowane = os.path.isdir(DANE) and folder_z_klasami(DANE)
    if not juz_rozpakowane:
        zipy = sorted(glob.glob(os.path.join(PROJEKT, "*.zip")))
        folder_na_dysku = folder_z_klasami(PROJEKT)
        if zipy:
            print("Rozpakowuję z Dysku:", zipy[0])
            shutil.unpack_archive(zipy[0], DANE)
        elif folder_na_dysku:
            print("Kopiuję zbiór z Dysku:", folder_na_dysku)
            shutil.copytree(folder_na_dysku, os.path.join(DANE, "rps-cv-images"))
        else:
            print("Pierwsze uruchomienie: pobieram zbiór z Kaggle...")
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=True)
            import kagglehub
            pobrane = kagglehub.dataset_download("drgfreeman/rockpaperscissors")
            shutil.copytree(pobrane, DANE, dirs_exist_ok=True)
            shutil.make_archive(os.path.join(PROJEKT, "rockpaperscissors"), "zip", pobrane)
            print("Zapisano kopię na Dysku: K10/rockpaperscissors.zip")

    DANE_COLAB = folder_z_klasami(DANE)
    print("Colab: zbiór =", DANE_COLAB, "| wyniki =", OUT_DIR_COLAB)
else:
    print("To nie jest Colab: pomijam Dysk Google.")

## 1. Ustawienia

In [ ]:
IMG = 96                 # wejście modelu (musi się zgadzać z MODEL_IMG na K10)
KADR = 240               # K10 wycina z kamery kwadrat 240x240 i zmniejsza go do 96x96
ALPHA = 0.35
BATCH = 32
EPOCHS_HEAD = 12
EPOCHS_FINE = 30
KLASY_DLONI = ["paper", "rock", "scissors"]
KLASA_BRAK = True        # dodatkowa klasa "none" = w kadrze NIE MA dłoni; K10 pokazuje wtedy "BRAK REKI" zamiast zgadywać gest
CLASSES = KLASY_DLONI + (["none"] if KLASA_BRAK else [])   # kolejność = kolejność etykiet na K10
BRAK = len(KLASY_DLONI)  # numer klasy "none"

# Folder, w którym są podfoldery paper/, rock/, scissors/.
# Na Kaggle: "/kaggle/input/rockpaperscissors/rps-cv-images".
# W lokalnym Jupyterze: ścieżka WZGLĘDEM FOLDERU NOTEBOOKA albo pełna ścieżka na dysku.
# Uwaga: "Copy Path" w JupyterLab daje ścieżkę względem folderu startowego Jupytera, nie notebooka.
# Jeśli ścieżka jest zła, notebook sam poszuka folderu z paper/rock/scissors w pobliżu.
KAGGLE_DIR = DANE_COLAB or "/kaggle/input/rockpaperscissors/rps-cv-images"
ROZMIAR_PULI = 60000     # ile zdjęć treningowych przygotować przed treningiem (więcej = dłużej, ale lepiej)
UDZIAL_ZDJEC = 0.5       # jaka część puli to prawdziwe zdjęcia ludzi (reszta to sztuczne sceny)
DODAJ_RENDERY = True     # dłonie renderowane komputerowo (białe tło): inne kolory skóry i ułożenia dłoni, 230 MB
DODAJ_ZDJECIA = True     # prawdziwe zdjęcia ludzi pokazujących gesty (HaGRID, 1 GB): najwięcej dają na płytce
SPRAWDZIAN_KAMERKA = True  # osobny zbiór zdjęć z kamerki internetowej TYLKO do sprawdzenia modelu (70 MB)
PROCESY = os.cpu_count() or 1   # ile procesów naraz przygotowuje zdjęcia (mniej = wolniej, ale mniej RAM-u)
ROZMIAR_DLONI = 240     # dłuższy bok przechowywanej wyciętej dłoni (mniej = mniej RAM)
LICZBA_ZDJEC_TLA = 1000   # ile zdjęć na tła trzymać w pamięci (mniej = mniej RAM)
CZESC_WALIDACJI = 0.15   # walidacja na dłoniach, których model nie widział w treningu
POBIERZ_ZDJECIA_TLA = True   # fragmenty zdjęć z flower_photos (TensorFlow) jako realistyczne tła
TLA_WNETRZA = "stepanyarullin/interior-design-styles"   # zbiór z Kaggle ze zdjęciami wnętrz ("" = wyłącz)
ZDJEC_NA_STYL = 50       # ile zdjęć wnętrz brać z każdej klasy (stylu) tego zbioru
BG_DIR = ""              # opcjonalnie: własny folder ze zdjęciami na tła (np. dataset wnętrz z Kaggle)
K10_TEST_DIR = ""        # opcjonalnie: kilka zdjęć z K10 (paper/rock/scissors) TYLKO do sprawdzenia modelu
SEED = 42
OUT_DIR = OUT_DIR_COLAB or ("/kaggle/working" if os.path.isdir("/kaggle/working") else ".")
FOLDER_DANYCH = os.path.join(os.path.expanduser("~"), ".cache", "k10_rps")   # tu trafiają pobrane zbiory dodatkowe

def ma_klasy(folder):
    return all(os.path.isdir(os.path.join(folder, c)) for c in KLASY_DLONI)

def znajdz_zbior(sciezka, max_glebokosc=6):
    kandydaci = [sciezka, sciezka.lstrip("/"), os.path.join(os.getcwd(), sciezka.lstrip("/"))]
    for k in kandydaci:
        if k and ma_klasy(k):
            return os.path.abspath(k)
    print(f"Nie ma folderu {sciezka!r} (bieżący folder: {os.getcwd()}). Szukam zbioru w pobliżu...")
    for baza in [os.getcwd(), os.path.dirname(os.getcwd()), os.path.expanduser("~"), "/kaggle/input"]:
        if not os.path.isdir(baza):
            continue
        for root, dirs, _ in os.walk(baza):
            dirs[:] = [d for d in dirs if not d.startswith(".") and d not in ("site-packages", "node_modules", "drive")]
            if root[len(baza):].count(os.sep) >= max_glebokosc:
                dirs[:] = []
            if ma_klasy(root):
                return os.path.abspath(root)
    raise FileNotFoundError("Nie znalazłam folderu z podfolderami paper/, rock/, scissors/. "
                            "Wpisz pełną ścieżkę w KAGGLE_DIR.")

def pobierz_i_rozpakuj(nazwa, url):
    """Pobiera archiwum do FOLDER_DANYCH/nazwa i je rozpakowuje (tylko za pierwszym razem). Zwraca folder."""
    cel = os.path.join(FOLDER_DANYCH, nazwa)
    if os.path.isdir(cel) and os.listdir(cel):
        return cel
    os.makedirs(cel, exist_ok=True)
    plik = os.path.join(cel, os.path.basename(url))
    print(f"Pobieram {nazwa} ...")
    urllib.request.urlretrieve(url, plik)
    shutil.unpack_archive(plik, cel)
    os.remove(plik)
    return cel

KAGGLE_DIR = znajdz_zbior(KAGGLE_DIR)
print("Zbiór:", KAGGLE_DIR)
for c in KLASY_DLONI:
    print(f"  {c}: {len(os.listdir(os.path.join(KAGGLE_DIR, c)))} plików")

def ustaw_seed(s):
    """Jedno źródło losowości dla całego generatora zdjęć (każdy proces roboczy dostaje własne)."""
    global rng
    rng = np.random.default_rng(s)
    random.seed(int(np.random.default_rng(s).integers(1 << 30)))

ustaw_seed(SEED)
tf.keras.utils.set_random_seed(SEED)

## 2. Dłonie do sztucznych scen

**Zielone tło (Kaggle).** Tło to piksele o zielonym odcieniu, wystarczającym nasyceniu i jasności. Z maski
zostawiamy tylko duże fragmenty (dłoń z ręką). Zielona poświata na brzegach dłoni jest usuwana (*despill*).

**Białe tło (dłonie renderowane, zbiór Laurence'a Moroneya, CC BY 2.0).** Tło jest tam idealnie szare lub
białe, więc wystarczy znaleźć bezbarwne, jasne piksele połączone z brzegiem obrazu. Te dłonie mają różne
kolory skóry i są pokazane z różnych stron, czego brakuje w zbiorze z zielonym tłem.

Każde wycięcie dostaje **czystą krawędź**: maska jest lekko zwężana, a pod jej brzegiem wpisywany jest
kolor pobliskiej skóry. Bez tego po skalowaniu wokół dłoni zostaje ciemna albo biała obwódka, po której
model mógłby rozpoznawać „wklejoną” dłoń zamiast jej kształtu. Na koniec każdą dłoń obracamy tak,
żeby ręka wchodziła od dołu.

In [ ]:
H_MIN, H_MAX = 35, 95    # zakres odcienia zieleni w OpenCV (0–180)
S_MIN, V_MIN = 60, 30

def wypelnij_kolor(rgb, rdzen):
    """Pod przezroczystymi pikselami i na samej krawędzi wpisuje kolor pobliskiej skóry."""
    m = (rdzen > 127).astype(np.float32)
    x = rgb.astype(np.float32)
    licz = cv2.GaussianBlur(x * m[..., None], (0, 0), 4)
    mian = cv2.GaussianBlur(m, (0, 0), 4)[..., None]
    sredni = (x * m[..., None]).sum((0, 1)) / max(m.sum(), 1.0)
    wypelnienie = np.where(mian > 1e-3, licz / np.maximum(mian, 1e-3), sredni)
    return np.where(m[..., None] > 0, x, wypelnienie).clip(0, 255).astype(np.uint8)

def najwieksze_fragmenty(maska, prog=0.15):
    """Zostawia tylko duże spójne fragmenty maski (usuwa drobinki i szum)."""
    n, lab, stats, _ = cv2.connectedComponentsWithStats(maska, 8)
    if n <= 1:
        return None
    najw = stats[1:, cv2.CC_STAT_AREA].max()
    keep = np.zeros_like(maska)
    for i in range(1, n):
        if stats[i, cv2.CC_STAT_AREA] >= prog * najw:
            keep[lab == i] = 255
    if keep.sum() / 255 < 0.03 * keep.size:      # prawie nic nie zostało -> pomiń
        return None
    return keep

def uloz_wyciecie(rgb, keep):
    """Wspólne wykończenie: czysta krawędź, ręka wchodzi od dołu, przycięcie, zmniejszenie. Zwraca RGBA."""
    rdzen = cv2.erode(keep, np.ones((5, 5), np.uint8))            # pewne wnętrze dłoni
    if rdzen.sum() == 0:
        return None
    rgb = wypelnij_kolor(rgb, rdzen)
    alfa = cv2.GaussianBlur(cv2.erode(keep, np.ones((3, 3), np.uint8)), (3, 3), 0)
    # obróć tak, żeby ręka (krawędź z największą liczbą pikseli dłoni) była na dole
    krawedzie = [keep[0, :].sum(), keep[:, -1].sum(), keep[-1, :].sum(), keep[:, 0].sum()]  # góra, prawo, dół, lewo
    k = {0: 2, 1: 3, 2: 0, 3: 1}[int(np.argmax(krawedzie))]    # liczba obrotów o 90° w lewo
    rgba = np.ascontiguousarray(np.rot90(np.dstack([rgb, alfa]), k))
    ys, xs = np.where(rgba[..., 3] > 0)
    if len(ys) == 0:
        return None
    rgba = rgba[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
    skala = ROZMIAR_DLONI / max(rgba.shape[:2])              # mały rozmiar = mało RAM-u
    if skala < 1:
        rgba = cv2.resize(rgba, None, fx=skala, fy=skala, interpolation=cv2.INTER_AREA)
    return rgba

def wytnij_dlon(bgr):
    """Dłoń z zielonego tła."""
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    h, s, v = cv2.split(hsv)
    tlo = (h >= H_MIN) & (h <= H_MAX) & (s >= S_MIN) & (v >= V_MIN)
    maska = (~tlo).astype(np.uint8) * 255
    maska = cv2.morphologyEx(maska, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    maska = cv2.morphologyEx(maska, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8))
    keep = najwieksze_fragmenty(maska)
    if keep is None:
        return None
    # despill: na dłoni zielony kanał nie może być mocniejszy niż max(R, B)
    b, g, r = cv2.split(bgr.astype(np.int16))
    g = np.minimum(g, np.maximum(r, b))
    return uloz_wyciecie(cv2.merge([r, g, b]).clip(0, 255).astype(np.uint8), keep)

def wytnij_dlon_biale(bgr):
    """Dłoń z białego tła (rendery). Tło jest tam idealnie szare/białe (R=G=B), a nawet bardzo jasna skóra ma odcień."""
    x = bgr.astype(np.int16)
    jasne = ((x.max(2) - x.min(2) <= 3) & (x.max(2) >= 200)).astype(np.uint8)
    n, lab, stats, _ = cv2.connectedComponentsWithStats(jasne, connectivity=4)
    brzeg = np.unique(np.concatenate([lab[0], lab[-1], lab[:, 0], lab[:, -1]]))
    duze = np.where(stats[:, cv2.CC_STAT_AREA] >= 25)[0]          # także prześwity między palcami
    etykiety = np.union1d(brzeg, duze)
    tlo = np.isin(lab, etykiety[etykiety != 0])                   # etykieta 0 = piksele, które nie są jasnoszare
    maska = cv2.morphologyEx((~tlo).astype(np.uint8) * 255, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    keep = najwieksze_fragmenty(maska)
    if keep is None:
        return None
    return uloz_wyciecie(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB), keep)

# dlonie[klasa] = lista par (grupa, wycięcie RGBA). Grupa = "ta sama ręka": przy podziale na trening
# i walidację cała grupa trafia w jedno miejsce, żeby walidacja była uczciwa.
dlonie = {c: [] for c in KLASY_DLONI}
for c in KLASY_DLONI:
    pliki = sorted(glob.glob(os.path.join(KAGGLE_DIR, c, "*")))
    for i, p in enumerate(pliki):
        img = cv2.imread(p)
        cut = wytnij_dlon(img) if img is not None else None
        if cut is not None:
            dlonie[c].append((f"zielone-{c}-{i}", cut))
    print(f"{c}: plików {len(pliki)}, wycięto {len(dlonie[c])} dłoni z zielonego tła")

if min(len(v) for v in dlonie.values()) == 0:
    raise RuntimeError("Dla którejś klasy nie wycięto żadnej dłoni. Jeśli 'plików' = 0, popraw KAGGLE_DIR. "
                       "Jeśli pliki są, a wycięto 0, zmień progi H_MIN/H_MAX/S_MIN/V_MIN.")

if DODAJ_RENDERY:
    try:
        ile = {c: 0 for c in KLASY_DLONI}
        for nazwa in ("rps", "rps-test-set"):
            folder = pobierz_i_rozpakuj(nazwa, f"https://storage.googleapis.com/download.tensorflow.org/data/{nazwa}.zip")
            for c in KLASY_DLONI:
                for p in sorted(glob.glob(os.path.join(folder, "**", c, "*.png"), recursive=True)):
                    img = cv2.imread(p)
                    cut = wytnij_dlon_biale(img) if img is not None else None
                    if cut is not None:
                        # w nazwie pliku (np. paper03-017.png) część przed myślnikiem to model dłoni
                        dlonie[c].append(("render-" + os.path.basename(p).split("-")[0], cut))
                        ile[c] += 1
        print("Dłonie renderowane:", ile)
    except Exception as e:
        print("Nie udało się pobrać dłoni renderowanych (zostają same dłonie z zielonego tła):", e)

Podgląd wycinania: czarne tło = to, co zostało uznane za tło. Po lewej dłonie z zielonego tła, po prawej renderowane.

In [ ]:
fig, ax = plt.subplots(3, 8, figsize=(16, 7))
for r, c in enumerate(KLASY_DLONI):
    zielone = [d for g, d in dlonie[c] if g.startswith("zielone")]
    rendery = [d for g, d in dlonie[c] if g.startswith("render")] or zielone
    for j in range(8):
        lista = zielone if j < 4 else rendery
        cut = lista[rng.integers(len(lista))]
        pod = (cut[..., :3].astype(np.float32) * (cut[..., 3:] / 255.0)).astype(np.uint8)
        ax[r, j].imshow(pod); ax[r, j].axis("off")
    ax[r, 0].set_title(c, loc="left")
plt.tight_layout(); plt.show()

## 2b. Prawdziwe zdjęcia dłoni

Sztuczne sceny nigdy nie wyglądają całkiem jak obraz z kamery, dlatego model uczy się też na prawdziwych
zdjęciach. Nie trzeba ich robić samemu:

- **HaGRID** (próbka 30 tys. zdjęć, licencja CC BY-SA 4.0): ludzie pokazujący gesty do kamerki w swoich domach.
  Każde zdjęcie ma zaznaczoną ramkę dłoni. Bierzemy gesty pasujące do gry: pięść = kamień, otwarta dłoń = papier,
  „V” i dwa palce = nożyce. Zdjęcia są dzielone **według osób**: 70% osób do treningu, 10% do walidacji,
  20% tylko do sprawdzianu.
- **Zdjęcia z kamerki internetowej** (zbiór z Roboflow Universe, CC BY 4.0): ludzie grający w kamień–papier–nożyce
  blisko kamery. Ten zbiór **nie trafia do treningu**. Służy tylko do sprawdzenia modelu na zdjęciach z zupełnie
  innego źródła.

- **Kadry bez dłoni** (klasa `none`, jeśli `KLASA_BRAK`): z tych samych zdjęć HaGRID wycinamy też kwadrat,
  w którym nie ma żadnej dłoni: twarz, ubranie, kawałek pokoju. Co drugi ma skalę wycinka dłoni, pozostałe są
  szersze. Podział według osób jest ten sam. Tak samo powstaje sprawdzian „bez dłoni” ze zdjęć z kamerki.

Z każdego zdjęcia zapamiętujemy kwadratowy wycinek wokół dłoni (bez wychodzenia poza zdjęcie). Wycinki leżą
w pamięci jako JPEG, bo tak zajmują kilka razy mniej RAM-u.

In [ ]:
BAZA = 176            # bok przechowywanego wycinka (piksele)
ZAPAS = 2.3           # wycinek ma bok = ZAPAS x dłuższy bok ramki dłoni (jeśli zdjęcie na to pozwala)
MIN_DLON_PX = 28      # mniejszych dłoni nie bierzemy: po powiększeniu byłyby rozmytą plamą

def spakuj(rgb):
    """Wycinek jako JPEG w pamięci (kilka razy mniej RAM-u). Rozpakowuje go kadr_ze_zdjecia."""
    return cv2.imencode(".jpg", cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, 95])[1]

def wycinek_dloni(rgb, ramka):
    """ramka = (x, y, w, h) w pikselach. Zwraca kwadratowy wycinek BAZAxBAZA wokół dłoni oraz to,
    gdzie w nim jest dłoń: (środek x, środek y, dłuższy bok) w ułamkach boku wycinka."""
    H, W = rgb.shape[:2]
    x, y, w, h = ramka
    bok = min(ZAPAS * max(w, h), H, W)
    cx = min(max(x + w / 2, bok / 2), W - bok / 2)
    cy = min(max(y + h / 2, bok / 2), H - bok / 2)
    s = BAZA / bok
    M = np.float32([[s, 0, BAZA / 2 - cx * s], [0, s, BAZA / 2 - cy * s]])
    wyc = cv2.warpAffine(rgb, M, (BAZA, BAZA), flags=cv2.INTER_AREA if s < 1 else cv2.INTER_LINEAR,
                         borderMode=cv2.BORDER_REPLICATE)
    return spakuj(wyc), np.float32((0.5 + (x + w / 2 - cx) / bok, 0.5 + (y + h / 2 - cy) / bok, max(w, h) / bok))

los_brak = np.random.default_rng(SEED)
def wycinek_bez_dloni(rgb, ramki, bok_od, bok_do):
    """Do klasy "none": kwadratowy wycinek zdjęcia, na którym nie ma żadnej dłoni (twarz, ubranie, pokój),
    o boku między bok_od a bok_do pikseli. Zwraca listę z jednym wycinkiem w tej samej postaci co wycinek_dloni,
    żeby dalej szedł tą samą drogą co wycinki z dłonią; pustą, jeśli na zdjęciu nie ma takiego miejsca."""
    H, W = rgb.shape[:2]
    for _ in range(20):
        bok = min(los_brak.uniform(bok_od, bok_do), H, W)
        x0, y0 = los_brak.uniform(0, W - bok), los_brak.uniform(0, H - bok)
        # ramki dłoni są ciasne, więc omijamy je z zapasem (nadgarstek też zdradza dłoń)
        if all(x0 > x + 1.3 * w or x0 + bok < x - 0.3 * w or y0 > y + 1.3 * h or y0 + bok < y - 0.3 * h
               for x, y, w, h in ramki):
            s = BAZA / bok
            wyc = cv2.warpAffine(rgb, np.float32([[s, 0, -x0 * s], [0, s, -y0 * s]]), (BAZA, BAZA),
                                 flags=cv2.INTER_AREA if s < 1 else cv2.INTER_LINEAR)
            return [(spakuj(wyc), np.float32((0.5, 0.5, 1 / ZAPAS)))]
    return []

# gest w HaGRID -> klasa w grze. "rock" w HaGRID to gest rockowy (rogi), więc go NIE bierzemy.
GESTY = {"fist": "rock", "palm": "paper", "stop": "paper", "stop_inverted": "paper",
         "peace": "scissors", "peace_inverted": "scissors", "two_up": "scissors", "two_up_inverted": "scissors"}
GESTY_PEWNE = {"fist", "palm", "stop", "stop_inverted", "peace", "peace_inverted"}   # tylko te idą do sprawdzianu

# zdjecia[część][numer klasy] = lista par (wycinek, położenie dłoni); części: train / val / test
zdjecia = {cz: {k: [] for k in range(len(CLASSES))} for cz in ("train", "val", "test")}
if DODAJ_ZDJECIA:
    try:
        folder = pobierz_i_rozpakuj("hagrid", "https://huggingface.co/datasets/cj-mills/hagrid-sample-30k-384p/"
                                              "resolve/main/hagrid-sample-30k-384p.zip")
        opisy = glob.glob(os.path.join(folder, "**", "ann_train_val"), recursive=True)[0]
        for gest, klasa in GESTY.items():
            ann = json.load(open(os.path.join(opisy, gest + ".json")))
            for p in sorted(glob.glob(os.path.join(os.path.dirname(opisy), "**", "train_val_" + gest, "*.jpg"), recursive=True)):
                a = ann.get(os.path.splitext(os.path.basename(p))[0])
                img = cv2.imread(p)
                if a is None or img is None:
                    continue
                rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                H, W = rgb.shape[:2]
                # ta sama osoba zawsze trafia do tej samej części
                b = int(hashlib.md5(a["user_id"].encode()).hexdigest(), 16) % 100
                czesc = "test" if b < 20 else ("val" if b < 30 else "train")
                if KLASA_BRAK:   # z tego samego zdjęcia wycinek bez dłoni: na zmianę w skali wycinka dłoni i szerszy
                    ramki = [(x * W, y * H, w * W, h * H) for x, y, w, h in a["bboxes"]]
                    bok = ZAPAS * max(max(r[2:]) for r in ramki)
                    zdjecia[czesc][BRAK] += wycinek_bez_dloni(rgb, ramki, bok, bok * (1 if len(zdjecia[czesc][BRAK]) % 2 else 2.5))
                if czesc == "test" and gest not in GESTY_PEWNE:
                    continue
                for (x, y, w, h), lab in zip(a["bboxes"], a["labels"]):
                    if lab == gest and max(w * W, h * H) >= MIN_DLON_PX:
                        zdjecia[czesc][CLASSES.index(klasa)].append(wycinek_dloni(rgb, (x * W, y * H, w * W, h * H)))
        for cz in zdjecia:
            print(f"HaGRID {cz}:", {c: len(zdjecia[cz][k]) for k, c in enumerate(CLASSES)})
    except Exception as e:
        print("Nie udało się pobrać zdjęć HaGRID (trening tylko na sztucznych scenach):", e)
MAM_ZDJECIA = all(len(zdjecia[cz][k]) > 0 for cz in ("train", "val") for k in range(len(CLASSES)))

# osobny sprawdzian: zdjęcia z kamerki internetowej (etykiety YOLO: 0 rock, 1 paper, 2 scissors)
kamerka, kamerka_bez = [], []
if SPRAWDZIAN_KAMERKA:
    try:
        folder = os.path.join(FOLDER_DANYCH, "kamerka")
        if not glob.glob(os.path.join(folder, "images", "*", "*.jpg")):
            try:
                from huggingface_hub import snapshot_download
            except ImportError:
                subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=True)
                from huggingface_hub import snapshot_download
            print("Pobieram zdjęcia z kamerki (ok. 3 tys. małych plików, kilka minut) ...")
            snapshot_download(repo_id="Xuban/rock-paper-scissors-roboflow", repo_type="dataset", local_dir=folder)
        for p in sorted(glob.glob(os.path.join(folder, "images", "*", "*.jpg"))):
            opis = p.replace(os.sep + "images" + os.sep, os.sep + "labels" + os.sep)[:-4] + ".txt"
            img = cv2.imread(p)
            if img is None or not os.path.exists(opis):
                continue
            rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            H, W = rgb.shape[:2]
            ramki = []
            for wiersz in open(opis).read().split("\n"):
                if len(wiersz.split()) < 5:
                    continue
                k, cx, cy, w, h = wiersz.split()[:5]
                cx, cy, w, h = float(cx) * W, float(cy) * H, float(w) * W, float(h) * H
                ramki.append((cx - w / 2, cy - h / 2, w, h))
                if max(w, h) >= MIN_DLON_PX:
                    kamerka.append((CLASSES.index(("rock", "paper", "scissors")[int(k)]), *wycinek_dloni(rgb, ramki[-1])))
            if KLASA_BRAK and ramki:   # tu dłonie są duże, więc bierzemy dowolny spory kwadrat, w którym ich nie ma
                kamerka_bez += [(BRAK, *w) for w in wycinek_bez_dloni(rgb, ramki, 0.35 * min(H, W), 0.8 * min(H, W))]
        print("Zdjęcia z kamerki (tylko sprawdzian):", {c: sum(k == i for k, _, _ in kamerka) for i, c in enumerate(KLASY_DLONI)},
              "| wycinki bez dłoni:", len(kamerka_bez))
    except Exception as e:
        print("Nie udało się pobrać zdjęć z kamerki (sprawdzian tylko na HaGRID):", e)

## 3. Tła

Mieszamy cztery źródła:
- **zdjęcia wnętrz** ze zbioru Kaggle [`interior-design-styles`](https://www.kaggle.com/datasets/stepanyarullin/interior-design-styles),
  po `ZDJEC_NA_STYL` (50) losowych zdjęć z każdego stylu. To najbardziej realistyczne tło, bo tak wygląda pokój za Twoją dłonią.
  Wybrane zdjęcia są zapisywane w folderze `tla_wnetrza` (w Colabie na Dysku: `K10/tla_wnetrza`),
  więc cały zbiór pobiera się tylko raz,
- **fragmenty zdjęć** `flower_photos` z TensorFlow (liście, niebo, ziemia),
- **Twój folder `BG_DIR`**, jeśli go podasz,
- **tła generowane**: ściana + blat, gradienty, rozmyte przedmioty, tkaniny.

In [ ]:
def lista_zdjec(folder):
    ext = ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG")
    return [p for e in ext for p in glob.glob(os.path.join(folder, "**", e), recursive=True)]

zdjecia_tla = []
if POBIERZ_ZDJECIA_TLA:
    try:
        p = keras.utils.get_file(
            "flower_photos.tgz",
            "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz",
            extract=True)
        zdjecia_tla += [f for f in lista_zdjec(os.path.dirname(p)) if "flower_photos" in f]
    except Exception as e:
        print("Nie udało się pobrać zdjęć na tła (zostają tła generowane):", e)
if BG_DIR and os.path.isdir(BG_DIR):
    zdjecia_tla += lista_zdjec(BG_DIR)
print("Zdjęć dostępnych jako tła:", len(zdjecia_tla))

# ---- zdjęcia wnętrz: po ZDJEC_NA_STYL z każdego stylu, zapisane na stałe w folderze tla_wnetrza ----
FOLDER_WNETRZ = os.path.join(PROJEKT if W_COLAB else OUT_DIR, "tla_wnetrza")

def pobierz_wnetrza():
    os.makedirs(FOLDER_WNETRZ, exist_ok=True)
    # na Kaggle zbiór może być dodany jako Input; wszędzie indziej pobieramy go przez kagglehub
    kandydaci = [d for d in glob.glob("/kaggle/input/*") if "interior" in d.lower()]
    if kandydaci:
        zrodlo = kandydaci[0]
    else:
        try:
            import kagglehub
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=True)
            import kagglehub
        print("Pobieram zbiór wnętrz z Kaggle (jednorazowo, może potrwać kilka minut)...")
        zrodlo = kagglehub.dataset_download(TLA_WNETRZA)
    # klasa = nazwa folderu, w którym leży zdjęcie (train/test z tymi samymi stylami są łączone)
    style = {}
    for f in lista_zdjec(zrodlo):
        style.setdefault(os.path.basename(os.path.dirname(f)).lower(), []).append(f)
    los = random.Random(SEED)
    for styl, pliki in sorted(style.items()):
        for i, f in enumerate(los.sample(sorted(pliki), min(ZDJEC_NA_STYL, len(pliki)))):
            shutil.copy(f, os.path.join(FOLDER_WNETRZ, f"{styl}_{i:02d}{os.path.splitext(f)[1].lower()}"))
    print(f"Wybrano zdjęcia wnętrz z {len(style)} stylów: {', '.join(sorted(style))}")

zdjecia_wnetrz = []
if TLA_WNETRZA:
    if not lista_zdjec(FOLDER_WNETRZ):
        try:
            pobierz_wnetrza()
        except Exception as e:
            print("Nie udało się pobrać zdjęć wnętrz (zostają pozostałe tła):", e)
            print("Jeśli to błąd logowania: w Colabie dodaj sekrety KAGGLE_USERNAME i KAGGLE_KEY "
                  "(ikona klucza po lewej), na Kaggle dodaj zbiór przez Add Input.")
    else:
        print("Zdjęcia wnętrz już są w", FOLDER_WNETRZ)
    zdjecia_wnetrz = lista_zdjec(FOLDER_WNETRZ)
print("Zdjęć wnętrz:", len(zdjecia_wnetrz))

# wczytujemy zdjęcia do pamięci (pomniejszone), żeby nie czytać dysku w kółko
def wczytaj(lista):
    out = []
    for p in lista:
        im = cv2.imread(p)
        if im is None or min(im.shape[:2]) < 64:
            continue
        s = 260 / min(im.shape[:2])
        im = cv2.resize(im, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
        out.append(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
    return out

cache_wnetrz = wczytaj(zdjecia_wnetrz)
cache_tla = wczytaj(random.sample(zdjecia_tla, min(LICZBA_ZDJEC_TLA, len(zdjecia_tla))))
print(f"W pamięci: {len(cache_wnetrz)} zdjęć wnętrz, {len(cache_tla)} innych zdjęć")

def losowy_kolor():
    # przewaga stonowanych barw, jak we wnętrzach
    base = rng.uniform(40, 230)
    return np.clip(base + rng.normal(0, 35, 3), 0, 255)

def tlo_zdjecie(cache):
    im = cache[rng.integers(len(cache))]
    s = rng.uniform(KADR, min(im.shape[:2]))
    y = rng.integers(0, im.shape[0] - int(s) + 1)
    x = rng.integers(0, im.shape[1] - int(s) + 1)
    crop = im[y:y + int(s), x:x + int(s)]
    out = cv2.resize(crop, (KADR, KADR), interpolation=cv2.INTER_AREA)
    if rng.random() < 0.5:
        out = cv2.GaussianBlur(out, (0, 0), rng.uniform(0.5, 3.0))  # tło poza ostrością
    return out

def tlo_generowane():
    t = rng.integers(5)
    yy, xx = np.mgrid[0:KADR, 0:KADR].astype(np.float32) / KADR
    if t == 0:      # ściana + blat
        horyzont = rng.uniform(0.3, 0.8)
        a, b = losowy_kolor(), losowy_kolor()
        out = np.where((yy > horyzont)[..., None], b, a).astype(np.float32)
        out += rng.normal(0, 6, out.shape)
    elif t == 1:    # gradient
        a, b = losowy_kolor(), losowy_kolor()
        kat = rng.uniform(0, 2 * np.pi)
        w = np.clip((xx * np.cos(kat) + yy * np.sin(kat) + 1) / 2, 0, 1)[..., None]
        out = a * (1 - w) + b * w
    elif t == 2:    # rozmyte przedmioty
        out = np.full((KADR, KADR, 3), losowy_kolor(), np.float32)
        for _ in range(rng.integers(3, 9)):
            c = tuple(float(v) for v in losowy_kolor())
            if rng.random() < 0.5:
                cv2.ellipse(out, (int(rng.integers(KADR)), int(rng.integers(KADR))),
                            (int(rng.integers(15, 90)), int(rng.integers(15, 90))),
                            float(rng.uniform(0, 180)), 0, 360, c, -1)
            else:
                x0, y0 = rng.integers(-40, KADR, 2)
                cv2.rectangle(out, (int(x0), int(y0)), (int(x0 + rng.integers(20, 140)), int(y0 + rng.integers(20, 140))), c, -1)
        out = cv2.GaussianBlur(out, (0, 0), rng.uniform(2, 7))
    elif t == 3:    # tkanina / paski
        a, b = losowy_kolor(), losowy_kolor()
        f = rng.uniform(4, 25)
        kat = rng.uniform(0, np.pi)
        w = (np.sin((xx * np.cos(kat) + yy * np.sin(kat)) * f * 2 * np.pi) + 1) / 2
        out = a * (1 - w[..., None]) + b * w[..., None]
        out += rng.normal(0, 8, out.shape)
    else:           # plamy o niskiej częstotliwości
        n = int(rng.integers(3, 10))
        out = cv2.resize(rng.uniform(30, 230, (n, n, 3)).astype(np.float32), (KADR, KADR),
                         interpolation=cv2.INTER_CUBIC)
    return np.clip(out, 0, 255).astype(np.uint8)

def losowe_tlo():
    r = rng.random()
    if cache_wnetrz and r < 0.5:          # 50% wnętrza
        return tlo_zdjecie(cache_wnetrz)
    if cache_tla and r < 0.75:            # 25% inne zdjęcia
        return tlo_zdjecie(cache_tla)
    return tlo_generowane()               # reszta: tła generowane

## 4. Składanie sceny, udawanie kamery K10 i pula zdjęć

**Sztuczna scena.** Dłoń jest układana w kadrze jednym przekształceniem (odbicie, pochylenie, obrót, skala,
położenie), tak żeby koniec ręki chował się za krawędzią kadru, a palce zostały w kadrze. Potem dostaje
światło: jasność i odcień częściowo dopasowane do tła, czasem światło z jednej strony, miękką krawędź,
czasem cień na tle albo rozmycie od ruchu. W tle czasem pojawiają się plamy w kolorze skóry
(jak twarz albo druga ręka), żeby model patrzył na kształt, a nie na kolor.

**Prawdziwe zdjęcie.** Z wycinka wokół dłoni losujemy kadr: dłoń zajmuje od ok. 43% do 100% boku kadru,
jest trochę przesunięta, czasem odbita i obrócona. Na zdjęciach ludzie trzymają dłoń pionowo, a w grze często
poziomo, dlatego połowa kadrów jest dodatkowo obracana o 90° albo 180°.

**Klasa `none`.** Sztuczna scena bez dłoni to samo tło, częściej z plamami w kolorze skóry. Prawdziwy kadr
bez dłoni powstaje z wycinka bez dłoni dokładnie tak samo jak kadr z dłonią, żeby modelu nie zdradzało nic
poza samą dłonią. Z tego samego powodu 40% wklejanych dłoni trafia na prawdziwy kadr bez dłoni zamiast na
zwykłe tło: w grze dłoń widać zwykle na tle gracza, więc twarz w kadrze nie może znaczyć „nie ma dłoni”.

**Kamera K10** jest udawana tak samo dla obu rodzajów zdjęć.

**Pula.** Wszystkie zdjęcia treningowe powstają raz, przed treningiem, na kilku rdzeniach procesora naraz
(`PROCESY`). Dzięki temu epoka treningu trwa sekundy zamiast minut i można trenować tyle epok, ile trzeba.

In [ ]:
_YY, _XX = np.mgrid[0:KADR, 0:KADR].astype(np.float32) / KADR - 0.5   # współrzędne pikseli od -0.5 do 0.5

def warstwa_dloni(dlon):
    """Układa dłoń w kadrze KADRxKADR tak, że ręka wchodzi od dołu. Zwraca warstwę RGBA."""
    h, w = dlon.shape[:2]
    rogi = np.float32([[0, 0], [w, 0], [w, h], [0, h]])
    p = rogi + rng.uniform(-0.10, 0.10, (4, 2)) * [w, h]          # perspektywa: dłoń pochylona w 3D
    if rng.random() < 0.5:
        p = p[[1, 0, 3, 2]]                                       # lewa / prawa ręka
    kat = np.deg2rad(rng.uniform(-40, 40))
    s = rng.uniform(0.50, 1.30) * KADR / max(h, w)                # dłuższy bok wycięcia = 50–130% kadru
    R = s * np.array([[np.cos(kat), -np.sin(kat)], [np.sin(kat), np.cos(kat)]])
    q = (p - [w / 2, h / 2]) @ R.T
    M = cv2.getPerspectiveTransform(rogi, q.astype(np.float32))
    # gdzie po przekształceniu jest koniec ręki (dolna krawędź wycięcia)
    xs = np.where(dlon[-1, :, 3] > 0)[0]
    x0, x1 = (xs.min(), xs.max()) if len(xs) else (0, w - 1)
    koniec = cv2.perspectiveTransform(np.float32([[[x0, h], [x1, h]]]), M)[0]
    minx, miny = q.min(0)
    maxx, maxy = q.max(0)
    wys = maxy - miny
    # koniec ręki chowamy za dolną krawędzią kadru, ale tak, żeby palce zostały w kadrze
    dy = KADR - min(koniec[:, 1].min(), maxy - rng.uniform(0.0, 0.2) * wys)
    dy = max(dy, -miny - 0.04 * wys)                              # czubki palców najwyżej lekko ucięte
    dy = min(dy, KADR - miny - 0.65 * wys)                        # widać co najmniej 65% wysokości
    dx = rng.uniform(0.25, 0.75) * KADR - (minx + maxx) / 2
    T = np.array([[1, 0, dx], [0, 1, dy], [0, 0, 1]], np.float64)
    src = cv2.GaussianBlur(dlon, (0, 0), 0.35 / s) if s < 0.6 else dlon   # mocne zmniejszenie: najpierw wygładź
    widac = dlon[..., 3] > 127
    sr = [float(v) for v in dlon[..., :3][widac].mean(0)] if widac.any() else [128.0] * 3
    return cv2.warpPerspective(src, T @ M.astype(np.float64), (KADR, KADR), flags=cv2.INTER_LINEAR,
                               borderMode=cv2.BORDER_CONSTANT, borderValue=(sr[0], sr[1], sr[2], 0))

def oswietl_dlon(rgb, alfa, tlo):
    """Światło na dłoni: dopasowanie do tła, własna jasność i odcień, światło z jednej strony."""
    x = rgb.astype(np.float32) / 255.0
    waga = alfa.sum()
    if waga > 1:
        sr_d = (x * alfa[..., None]).sum((0, 1)) / waga
        sr_t = tlo.reshape(-1, 3).mean(0) / 255.0
        x = x * np.clip((sr_t + 0.08) / (sr_d + 0.08), 0.6, 1.5) ** rng.uniform(0.0, 0.5)
    x = x * rng.uniform(0.9, 1.1, 3) * rng.uniform(0.75, 1.25)
    if rng.random() < 0.6:                                        # światło z boku / z góry
        kat = rng.uniform(0, 2 * np.pi)
        x = x * (1 + rng.uniform(0.15, 0.6) * 2 * (_XX * np.cos(kat) + _YY * np.sin(kat)))[..., None]
    return np.clip(x, 0, 1) ** rng.uniform(0.8, 1.3) * 255.0

def rozmycie_ruchu(img, dlugosc, kat):
    k = np.zeros((dlugosc, dlugosc), np.float32)
    c = (dlugosc - 1) / 2
    a, b = c * np.cos(kat), c * np.sin(kat)
    cv2.line(k, (int(round(c - a)), int(round(c - b))), (int(round(c + a)), int(round(c + b))), 1.0, 1)
    return cv2.filter2D(img, -1, k / k.sum())

def dodaj_rozpraszacze(tlo):
    """Plamy w kolorze skóry w tle (twarz, druga ręka, ramię)."""
    out = tlo.astype(np.float32)
    for _ in range(rng.integers(1, 3)):
        warstwa = np.zeros((KADR, KADR), np.float32)
        cv2.ellipse(warstwa, (int(rng.integers(KADR)), int(rng.integers(KADR))),
                    (int(rng.integers(18, 80)), int(rng.integers(18, 80))), float(rng.uniform(0, 180)), 0, 360, 1.0, -1)
        warstwa = cv2.GaussianBlur(warstwa, (0, 0), rng.uniform(2, 8))[..., None] * rng.uniform(0.6, 1.0)
        kolor = KOLORY_SKORY[rng.integers(len(KOLORY_SKORY))] * rng.uniform(0.6, 1.15)
        out = out * (1 - warstwa) + kolor * warstwa
    return out

def scena(dlon, czesc="train"):
    """Sztuczna scena: float32 RGB 240x240. dlon = None daje scenę bez dłoni (klasa "none")."""
    if dlon is not None and KLASA_BRAK and zdjecia[czesc][BRAK] and rng.random() < 0.4:
        # dłoń na tle prawdziwego kadru bez dłoni (twarz, ubranie, pokój): w grze dłoń widać zwykle na tle gracza,
        # a model nie może się nauczyć, że "widać twarz" znaczy "nie ma dłoni"
        lista = zdjecia[czesc][BRAK]
        tlo = kadr_ze_zdjecia(*lista[rng.integers(len(lista))]).astype(np.float32)
    else:
        tlo = losowe_tlo().astype(np.float32)
    if rng.random() < (0.25 if dlon is not None else 0.5):
        tlo = dodaj_rozpraszacze(tlo)
    if dlon is None:
        return tlo
    warstwa = warstwa_dloni(dlon)
    k = rng.choice([0, 1, 3, 2], p=[0.4, 0.25, 0.25, 0.1])       # ręka od dołu, z boków, z góry
    if k:
        warstwa = np.ascontiguousarray(np.rot90(warstwa, k))
    alfa = warstwa[..., 3]
    if rng.random() < 0.5:
        alfa = cv2.erode(alfa, np.ones((3, 3), np.uint8))
    alfa = cv2.GaussianBlur(alfa, (0, 0), rng.uniform(0.6, 2.0)).astype(np.float32) / 255.0   # miękka krawędź
    rgb = oswietl_dlon(warstwa[..., :3], alfa, tlo)
    if rng.random() < 0.3:                                        # cień dłoni na tle
        M = np.float32([[1, 0, rng.uniform(-25, 25)], [0, 1, rng.uniform(-25, 25)]])
        cien = cv2.GaussianBlur(cv2.warpAffine(alfa, M, (KADR, KADR)), (0, 0), rng.uniform(4, 15))
        tlo = tlo * (1 - rng.uniform(0.2, 0.6) * cien[..., None])
    if rng.random() < 0.12:                                       # dłoń w ruchu, tło ostre
        dl, kat = int(rng.integers(5, 16)), rng.uniform(0, np.pi)
        rgb, alfa = rozmycie_ruchu(rgb, dl, kat), rozmycie_ruchu(alfa, dl, kat)
    return tlo * (1 - alfa[..., None]) + rgb * alfa[..., None]

def kadr_ze_zdjecia(wyc, gdzie, losowo=True, udzial=0.63):
    """Z wycinka prawdziwego zdjęcia robi kadr 240x240, w którym dłoń zajmuje `udzial` boku kadru
    (jeśli zdjęcie na to pozwala). Losowo: różna wielkość i położenie dłoni, odbicie, obrót (także o 90° i 180°)."""
    wyc = cv2.cvtColor(cv2.imdecode(wyc, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)   # wycinki leżą w pamięci jako JPEG
    hx, hy, hs = (float(v) for v in gdzie)
    if losowo:
        udzial = rng.uniform(0.43, 1.0)
        # na zdjęciach dłoń jest zwykle pionowo; w grze bywa też poziomo albo palcami w dół
        kat = rng.uniform(-20, 20) + rng.choice([0, 90, -90, 180], p=[0.5, 0.2, 0.2, 0.1])
    else:
        kat = 0.0
    z = max(1.0, udzial / hs)                               # powiększenie względem całego wycinka
    okno = 1.0 / z
    if losowo:
        luz = max(0.0, (okno - 0.9 * hs) / 2)               # dłoń może się przesunąć, ale zostaje w kadrze
        hx, hy = hx + rng.uniform(-luz, luz), hy + rng.uniform(-luz, luz)
    cx = min(max(hx, okno / 2), 1 - okno / 2) * BAZA        # okno nie wychodzi poza wycinek
    cy = min(max(hy, okno / 2), 1 - okno / 2) * BAZA
    M = cv2.getRotationMatrix2D((cx, cy), kat, z * KADR / BAZA)
    M[0, 2] += KADR / 2 - cx
    M[1, 2] += KADR / 2 - cy
    out = cv2.warpAffine(wyc, M, (KADR, KADR), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)
    if losowo and rng.random() < 0.5:
        out = out[:, ::-1]
    return np.ascontiguousarray(out)

def rgb565(x):
    """Redukcja do RGB565 i z powrotem do 8 bitów (r*255/31 itd.), jak na płytce."""
    r = np.round((x[..., 0] >> 3).astype(np.float32) * 255 / 31)
    g = np.round((x[..., 1] >> 2).astype(np.float32) * 255 / 63)
    b = np.round((x[..., 2] >> 3).astype(np.float32) * 255 / 31)
    return np.stack([r, g, b], -1).astype(np.uint8)

def udawaj_k10(img):
    """img: float32 RGB 240x240 -> uint8 RGB 96x96, jak po przetworzeniu na płytce."""
    x = img / 255.0
    x = x * rng.uniform(0.8, 1.2, 3)                        # balans bieli
    x = np.clip(x * rng.uniform(0.55, 1.35), 0, 1)          # ekspozycja
    x = x ** rng.uniform(0.7, 1.5)                          # gamma / krzywa sensora
    m = x.mean()
    x = np.clip((x - m) * rng.uniform(0.7, 1.15) + m, 0, 1) # kontrast
    if rng.random() < 0.3:                                  # niższe nasycenie
        g = x.mean(axis=2, keepdims=True)
        x = g + (x - g) * rng.uniform(0.4, 0.9)
    if rng.random() < 0.4:                                  # winietowanie: ciemniejsze rogi
        x = x * (1 - rng.uniform(0.1, 0.45) * np.clip((_XX ** 2 + _YY ** 2) * 2, 0, 1))[..., None]
    x = (x * 255).astype(np.float32)
    if rng.random() < 0.2:                                  # poruszona kamera
        x = rozmycie_ruchu(x, int(rng.integers(3, 12)), rng.uniform(0, np.pi))
    if rng.random() < 0.7:
        x = cv2.GaussianBlur(x, (0, 0), rng.uniform(0.4, 1.6))  # optyka / ostrość
    x = x + rng.normal(0, rng.uniform(1, 9), x.shape)           # szum sensora
    x = np.clip(x, 0, 255).astype(np.uint8)
    x = cv2.resize(x, (IMG, IMG), interpolation=cv2.INTER_AREA)  # uśrednianie bloków, jak w K10
    return rgb565(x)

def jak_k10(rgb240):
    """Tylko to, co płytka robi zawsze: zmniejszenie 240 -> 96 i RGB565. Do sprawdzianów."""
    return rgb565(cv2.resize(rgb240, (IMG, IMG), interpolation=cv2.INTER_AREA))

# ---- podział dłoni do sztucznych scen na trening / walidację: WEDŁUG GRUP (żeby walidacja była uczciwa) ----
dlonie_tr, dlonie_val = {}, {}
for c in KLASY_DLONI:
    grupy = sorted({g for g, _ in dlonie[c]})
    random.Random(SEED).shuffle(grupy)
    do_val, ile = set(), 0
    for g in grupy:                              # dobieramy całe grupy, aż będzie ok. CZESC_WALIDACJI dłoni
        if ile >= CZESC_WALIDACJI * len(dlonie[c]):
            break
        do_val.add(g)
        ile += sum(1 for gg, _ in dlonie[c] if gg == g)
    dlonie_val[c] = [d for g, d in dlonie[c] if g in do_val]
    dlonie_tr[c] = [d for g, d in dlonie[c] if g not in do_val]
    print(f"{c}: dłoni do scen: trening {len(dlonie_tr[c])}, walidacja {len(dlonie_val[c])}")

KOLORY_SKORY = [d[..., :3][d[..., 3] > 127].mean(0) for c in KLASY_DLONI for d in dlonie_tr[c][::7]]

# ---- jedno zdjęcie treningowe / walidacyjne ----
def probka(zadanie):
    rodzaj, k, czesc = zadanie
    if rodzaj == "zdjecie":
        lista = zdjecia[czesc][k]
        return udawaj_k10(kadr_ze_zdjecia(*lista[rng.integers(len(lista))]).astype(np.float32))
    lista = (dlonie_tr if czesc == "train" else dlonie_val).get(CLASSES[k])      # klasa "none" nie ma listy dłoni
    return udawaj_k10(scena(lista[rng.integers(len(lista))] if lista else None, czesc))

def paczka(zadania):
    return np.stack([probka(z) for z in zadania])

def start_procesu(seed):
    cv2.setNumThreads(0)                         # OpenCV w procesie roboczym ma liczyć na jednym wątku
    ustaw_seed([seed, mp.current_process()._identity[0]])

def generuj(n, czesc, seed, udzial_zdjec):
    """n zdjęć z równą liczbą każdej klasy; `udzial_zdjec` z nich to prawdziwe zdjęcia."""
    los = np.random.default_rng(seed)
    klasy = los.permutation(np.arange(n) % len(CLASSES))
    prawdziwe = los.random(n) < (udzial_zdjec if MAM_ZDJECIA else 0.0)
    zadania = [("zdjecie" if p else "scena", int(k), czesc) for p, k in zip(prawdziwe, klasy)]
    paczki = [zadania[i:i + 64] for i in range(0, n, 64)]
    co_ile = max(1, len(paczki) // 5)
    t0, X = time.time(), np.empty((n, IMG, IMG, 3), np.uint8)    # paczki trafiają od razu do jednej tablicy
    if PROCESY > 1 and "fork" in mp.get_all_start_methods():      # Linux / Colab / Kaggle: kilka procesów naraz
        with mp.get_context("fork").Pool(PROCESY, initializer=start_procesu, initargs=(seed,)) as pool:
            for i, w in enumerate(pool.imap(paczka, paczki)):
                X[i * 64:i * 64 + len(w)] = w
                if (i + 1) % co_ile == 0:
                    print(f"  {czesc}: {min((i + 1) * 64, n)}/{n} zdjęć, {time.time() - t0:.0f} s")
    else:                                                         # Windows / macOS: jeden proces
        ustaw_seed(seed)
        for i, p in enumerate(paczki):
            X[i * 64:i * 64 + len(p)] = paczka(p)
            if (i + 1) % co_ile == 0:
                print(f"  {czesc}: {min((i + 1) * 64, n)}/{n} zdjęć, {time.time() - t0:.0f} s")
    return X, klasy.astype(np.int32)

X_train, y_train = generuj(ROZMIAR_PULI, "train", SEED + 1, UDZIAL_ZDJEC)
X_val, y_val = generuj(3000, "val", SEED + 2, UDZIAL_ZDJEC)
ustaw_seed(SEED + 3)
print("Trening:", X_train.shape, "| walidacja:", X_val.shape)

# ---- sprawdziany na prawdziwych zdjęciach spoza treningu (bez losowych zmian, tylko to, co robi płytka) ----
SPRAWDZIANY = {}
def sprawdzian(lista_z_klasa):
    X = np.stack([jak_k10(kadr_ze_zdjecia(wyc, gdzie, losowo=False)) for _, wyc, gdzie in lista_z_klasa])
    return X, np.array([k for k, _, _ in lista_z_klasa], np.int32)
inne_osoby = [(k, wyc, gdzie) for k in range(len(KLASY_DLONI)) for wyc, gdzie in zdjecia["test"][k]]
if inne_osoby:
    SPRAWDZIANY["HaGRID, osoby spoza treningu"] = sprawdzian(inne_osoby)
if kamerka:
    SPRAWDZIANY["zdjęcia z kamerki (inny zbiór)"] = sprawdzian(kamerka)
# kadry bez dłoni osobno: tu jedyna dobra odpowiedź to "none"
if KLASA_BRAK and zdjecia["test"][BRAK]:
    SPRAWDZIANY["HaGRID bez dłoni, osoby spoza treningu"] = sprawdzian([(BRAK, *w) for w in zdjecia["test"][BRAK]])
if kamerka_bez:
    SPRAWDZIANY["kamerka bez dłoni (inny zbiór)"] = sprawdzian(kamerka_bez)
for nazwa, (X, y) in SPRAWDZIANY.items():
    print(f"Sprawdzian „{nazwa}”: {len(y)} zdjęć")

Podgląd: tak mniej więcej model zobaczy obraz z K10. Dwa górne rzędy to sztuczne sceny, dwa dolne to prawdziwe zdjęcia
(jeśli są). Warto przejrzeć, czy dłonie są dobrze wycięte i czy gest da się rozpoznać.

In [ ]:
fig, ax = plt.subplots(4, 8, figsize=(14, 7.5))
for i, a in enumerate(ax.flat):
    k = int(rng.integers(len(CLASSES)))
    rodzaj = "zdjecie" if (i >= 16 and MAM_ZDJECIA) else "scena"
    a.imshow(probka((rodzaj, k, "train"))); a.set_title(CLASSES[k], fontsize=9); a.axis("off")
plt.tight_layout(); plt.show()

## 5. Dane dla Keras (+ lekka augmentacja w locie)

Pula leży w pamięci w jednym egzemplarzu. Co epokę losujemy kolejność zdjęć, a każda paczka dostaje jeszcze
drobne losowe zmiany (odbicie, obrót, powiększenie, przesunięcie).

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
onehot = lambda y: tf.one_hot(y, len(CLASSES))

aug = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.04),
    layers.RandomZoom((-0.05, 0.1)),
    layers.RandomTranslation(0.05, 0.05),
])

def przygotuj(x, y):
    return tf.cast(x, tf.float32), onehot(y)

def paczki_treningowe():
    """Jedna epoka: wszystkie zdjęcia z puli w losowej kolejności, od razu w paczkach."""
    kolejnosc = np.random.permutation(len(y_train))
    for i in range(0, len(kolejnosc), BATCH):
        j = kolejnosc[i:i + BATCH]
        yield X_train[j], y_train[j]

train_aug = (tf.data.Dataset.from_generator(
                 paczki_treningowe,
                 output_signature=(tf.TensorSpec((None, IMG, IMG, 3), tf.uint8), tf.TensorSpec((None,), tf.int32)))
             .apply(tf.data.experimental.assert_cardinality(math.ceil(len(y_train) / BATCH)))   # pasek postępu zna długość epoki
             .map(przygotuj, num_parallel_calls=AUTOTUNE)
             .map(lambda x, y: (aug(x, training=True), y), num_parallel_calls=AUTOTUNE)
             .prefetch(AUTOTUNE))
val_b = tf.data.Dataset.from_tensor_slices((X_val, y_val)).map(przygotuj).batch(BATCH).prefetch(AUTOTUNE)

## 6. Model (tak samo jak w zwykłym notebooku)

In [ ]:
base = keras.applications.MobileNetV2(
    input_shape=(IMG, IMG, 3), alpha=ALPHA, include_top=False, weights="imagenet")
base.trainable = False

inputs = keras.Input((IMG, IMG, 3))
x = layers.Rescaling(1 / 127.5, offset=-1)(inputs)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(len(CLASSES), activation="softmax")(x)
model = keras.Model(inputs, outputs)

## 7. Trening: etap 1 (głowica) i etap 2 (dostrajanie)

In [ ]:
cb = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)]
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="categorical_crossentropy", metrics=["accuracy"])
model.fit(train_aug, validation_data=val_b, epochs=EPOCHS_HEAD, callbacks=cb)

base.trainable = True
for layer in base.layers[:-40]:
    layer.trainable = False
for layer in base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False
model.compile(optimizer=keras.optimizers.Adam(1e-4), loss="categorical_crossentropy", metrics=["accuracy"])
model.fit(train_aug, validation_data=val_b, epochs=EPOCHS_FINE, callbacks=cb)

model.save(os.path.join(OUT_DIR, "model.keras"))   # kopia wytrenowanego modelu (na Colabie: na Dysku)
print("Zapisano model.keras w", OUT_DIR)

In [ ]:
es = cb[0]   # EarlyStopping z komórki treningu
if es.best_weights is not None:
    model.set_weights(es.best_weights)
    print(f"Przywrócono wagi z najlepszej epoki dostrajania: {es.best_epoch + 1} (val_loss = {es.best:.4f})")
else:
    print("Brak zapamiętanych wag, zostają obecne")

model.save(os.path.join(OUT_DIR, "model.keras"))
print("Zapisano model.keras w", OUT_DIR)

## 8. Ocena (float32)

Najpierw walidacja (sztuczne sceny z nieznanych dłoni + zdjęcia nieznanych osób, z udawaną kamerą).
Potem **sprawdziany na prawdziwych zdjęciach spoza treningu**. Te liczby są najbliższe temu, co zobaczysz na płytce.
W sprawdzianach z dłońmi odpowiedź `none` liczy się jako błąd; w sprawdzianach „bez dłoni” to jedyna dobra odpowiedź.

In [ ]:
y_pred = np.argmax(model.predict(val_b, verbose=0), 1)
numery = list(range(len(CLASSES)))    # w sprawdzianach nie każda klasa występuje, więc podajemy je jawnie
print("Walidacja:")
print(classification_report(y_val, y_pred, labels=numery, target_names=CLASSES, digits=3, zero_division=0))
print(confusion_matrix(y_val, y_pred, labels=numery))

for nazwa, (X, y) in SPRAWDZIANY.items():
    p = np.argmax(model.predict(X.astype(np.float32), batch_size=256, verbose=0), 1)
    print(f"\nSprawdzian „{nazwa}”: trafność {np.mean(p == y):.3f} ({len(y)} zdjęć)")
    print(classification_report(y, p, labels=numery, target_names=CLASSES, digits=3, zero_division=0))
    print(confusion_matrix(y, p, labels=numery))

## 9. Kwantyzacja INT8

In [ ]:
import tensorflow as tf
import keras
from keras import layers
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
def representative_data():
    # 300 losowych zdjęć z puli treningowej (sztuczne sceny i prawdziwe zdjęcia)
    for i in np.random.default_rng(SEED).choice(len(X_train), 300, replace=False):
        yield [X_train[i][None].astype(np.float32)]

def do_int8(m):
    """Zamienia model Keras na model TFLite liczący w całości na liczbach 8-bitowych."""
    conv = tf.lite.TFLiteConverter.from_keras_model(m)
    conv.optimizations = [tf.lite.Optimize.DEFAULT]
    conv.representative_dataset = representative_data
    conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    conv.inference_input_type = tf.int8
    conv.inference_output_type = tf.int8
    # TFLite Micro na K10 nie obsługuje kwantyzacji "per-channel" w warstwie Dense
    conv._experimental_disable_per_channel_quantization_for_dense_layers = True
    return conv.convert()

def uruchom_int8(tfl, X):
    """Uruchamia model INT8 na zdjęciach uint8 (po jednym) i zwraca jego wyjścia jako zwykłe liczby."""
    it = tf.lite.Interpreter(model_content=tfl)
    it.allocate_tensors()
    we, wy = it.get_input_details()[0], it.get_output_details()[0]
    (s_we, z_we), (s_wy, z_wy) = we["quantization"], wy["quantization"]
    wynik = np.empty((len(X), wy["shape"][-1]), np.float32)
    for k in range(len(X)):
        q = np.clip(np.round(X[k:k + 1].astype(np.float32) / s_we + z_we), -128, 127).astype(np.int8)
        it.set_tensor(we["index"], q)
        it.invoke()
        wynik[k] = (it.get_tensor(wy["index"])[0].astype(np.float32) - z_wy) * s_wy
    return wynik

tflite_model = do_int8(model)
acc_przed = np.mean(uruchom_int8(tflite_model, X_val).argmax(1) == y_val)
with open(os.path.join(OUT_DIR, "rps_int8.tflite"), "wb") as f:
    f.write(tflite_model)
print(f"Model INT8: {len(tflite_model) / 1024:.0f} KB | trafność na walidacji: {acc_przed:.3f}")

## 9b. Dopasowanie ostatniej warstwy do modelu INT8

Po kwantyzacji wnętrze sieci liczy trochę inaczej niż w treningu, więc ostatnia warstwa (ta, która z cech wybiera
kamień, papier albo nożyce) dostaje nieco inne liczby niż te, na których się uczyła. To kosztuje kilka punktów
procentowych trafności. Naprawiamy to tanio: przepuszczamy zdjęcia treningowe przez **skwantyzowaną** sieć bez
ostatniej warstwy, na tak otrzymanych cechach douczamy samą ostatnią warstwę i kwantyzujemy model jeszcze raz.
Reszta sieci się nie zmienia, rozmiar modelu też nie. Jeśli walidacja nie wyjdzie lepiej, zostaje model sprzed
dopasowania.

In [ ]:
ostatnia = [l for l in model.layers if isinstance(l, layers.Dense)][-1]
wagi_przed = ostatnia.get_weights()
cechy_int8 = do_int8(keras.Model(model.input, ostatnia.input))      # ta sama sieć bez ostatniej warstwy
wybrane = np.random.default_rng(SEED).choice(len(X_train), min(15000, len(X_train)), replace=False)
F_train, F_val = uruchom_int8(cechy_int8, X_train[wybrane]), uruchom_int8(cechy_int8, X_val)

glowica = keras.Sequential([keras.Input((F_train.shape[1],)), layers.Dropout(0.3),
                            layers.Dense(len(CLASSES), activation="softmax")])
glowica.layers[-1].set_weights(wagi_przed)                           # start od obecnych wag
glowica.compile(optimizer=keras.optimizers.Adam(1e-4), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
glowica.fit(F_train, y_train[wybrane], validation_data=(F_val, y_val), epochs=60, batch_size=64, verbose=0,
            callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True)])

# Uwaga: od tej chwili model w pamięci ma ostatnią warstwę dobraną pod INT8. Plik model.keras zapisany wcześniej
# zostaje wersją sprzed dopasowania.
ostatnia.set_weights(glowica.layers[-1].get_weights())
tflite_po = do_int8(model)
acc_po = np.mean(uruchom_int8(tflite_po, X_val).argmax(1) == y_val)
print(f"INT8, trafność na walidacji: przed dopasowaniem {acc_przed:.3f}, po dopasowaniu {acc_po:.3f}")
if acc_po > acc_przed:
    tflite_model = tflite_po
    with open(os.path.join(OUT_DIR, "rps_int8.tflite"), "wb") as f:
        f.write(tflite_model)
    print("Zostaje model po dopasowaniu.")
else:
    ostatnia.set_weights(wagi_przed)
    print("Dopasowanie nie pomogło: zostaje model sprzed dopasowania.")

## 10. Sprawdzenie modelu INT8

To jest dokładnie ten model, który trafi na płytkę. Sprawdzamy go na walidacji i na tych samych
prawdziwych zdjęciach spoza treningu co wyżej. Jeśli podasz `K10_TEST_DIR`, także na Twoich zdjęciach z K10:
to najlepszy możliwy sprawdzian, nawet kilkanaście zdjęć na gest dużo mówi.

In [ ]:
interp = tf.lite.Interpreter(model_content=tflite_model)
interp.allocate_tensors()
inp, out = interp.get_input_details()[0], interp.get_output_details()[0]
in_scale, in_zp = inp["quantization"]
print("Operacje w modelu:", sorted({op["op_name"] for op in interp._get_ops_details()} - {"DELEGATE"}))

def int8_predict(img_uint8):
    q = np.clip(np.round(img_uint8.astype(np.float32) / in_scale + in_zp), -128, 127).astype(np.int8)
    interp.set_tensor(inp["index"], q[None])
    interp.invoke()
    return int(np.argmax(interp.get_tensor(out["index"])[0]))

acc = np.mean([int8_predict(x) == y for x, y in zip(X_val, y_val)])
print(f"INT8, walidacja: {acc:.3f}")
for nazwa, (X, y) in SPRAWDZIANY.items():
    p = np.array([int8_predict(x) for x in X])
    print(f"INT8, sprawdzian „{nazwa}”: {np.mean(p == y):.3f}  "
          + ", ".join(f"{c} {np.mean(p[y == i] == i):.2f}" for i, c in enumerate(CLASSES) if np.any(y == i)))

if K10_TEST_DIR and os.path.isdir(K10_TEST_DIR):
    ok = n = 0
    for ci, c in enumerate(CLASSES):
        for p in lista_zdjec(os.path.join(K10_TEST_DIR, c)):
            im = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
            h, w = im.shape[:2]; s = min(h, w)
            im = im[(h - s) // 2:(h - s) // 2 + s, (w - s) // 2:(w - s) // 2 + s]
            im = cv2.resize(im, (IMG, IMG), interpolation=cv2.INTER_AREA)
            ok += int8_predict(im) == ci; n += 1
    if n:
        print(f"INT8, PRAWDZIWE zdjęcia z K10: {ok / n:.3f}  ({ok}/{n})")

## 11. Eksport `model_data.h`

Pobierz `model_data.h` z zakładki *Output* i podmień nim plik w folderze `3_k10_rps/`.

In [ ]:
def write_header(data, path):
    lines = ["  " + ", ".join(f"0x{b:02x}" for b in data[i:i + 16]) + "," for i in range(0, len(data), 16)]
    labels = ", ".join(f'"{c}"' for c in CLASSES)
    with open(path, "w") as f:
        f.write("// Wygenerowane automatycznie przez notebook trenujący. Nie edytuj ręcznie.\n")
        f.write("#pragma once\n#include <stdint.h>\n\n")
        f.write(f"#define MODEL_IMG {IMG}\n#define MODEL_NUM_CLASSES {len(CLASSES)}\n")
        f.write(f"static const char* const MODEL_LABELS[MODEL_NUM_CLASSES] = {{{labels}}};\n\n")
        f.write("alignas(16) const uint8_t g_model[] = {\n" + "\n".join(lines) + "\n};\n")
        f.write(f"const unsigned int g_model_len = {len(data)};\n")

write_header(tflite_model, os.path.join(OUT_DIR, "model_data.h"))
print("Zapisano", os.path.join(OUT_DIR, "model_data.h"))